# Step 2: Reproducing the Original Literature (Baseline Model)

**Goal:** build a basic classification model and compare it against the
original paper `chicco2020machine` — Chicco & Jurman (2020), *BMC Medical
Informatics and Decision Making* — Random Forest, Accuracy 74.0%, MCC
0.384 (12 features, excluding `time`).

In [1]:
import sys
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, matthews_corrcoef, f1_score, classification_report

def _load_heart_failure_data():
    """Load the local file (../data/...) if present (running inside a
    cloned HMYT repo); otherwise (opened standalone via Colab/Kaggle, no
    accompanying data/ folder) automatically download it from the public
    mirror on hmyt-book (Public repo, verified 2026-09-23)."""
    import os
    local_path = "../data/heart_failure_clinical_records_dataset.csv"
    remote_url = ("https://raw.githubusercontent.com/fossbk-spec/hmyt-book/gh-pages/"
                  "labs_chuyen_de/ch02_suy_tim_risk_dxai/data/"
                  "heart_failure_clinical_records_dataset.csv")
    path = local_path if os.path.exists(local_path) else remote_url
    if path == remote_url:
        print(f"[i] Local data not found — downloading from the public mirror:\n    {remote_url}")
    return pd.read_csv(path).rename(columns={'death_event': 'DEATH_EVENT'})

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

df = _load_heart_failure_data()
FEATURE_COLS = [c for c in df.columns if c != 'DEATH_EVENT']
X, y = df[FEATURE_COLS], df['DEATH_EVENT'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print(f"Train: {X_train.shape[0]}  Test: {X_test.shape[0]}")


Train: 239  Test: 60


## 1. Logistic Regression (linear reference model)

In [2]:
logreg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
logreg.fit(X_train_s, y_train)
y_pred_lr = logreg.predict(X_test_s)

acc_lr = accuracy_score(y_test, y_pred_lr)
mcc_lr = matthews_corrcoef(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr)
print(f"Logistic Regression -> Accuracy: {acc_lr:.4f} | MCC: {mcc_lr:.4f} | F1: {f1_lr:.4f}")

Logistic Regression -> Accuracy: 0.8167 | MCC: 0.5563 | F1: 0.6667


## 2. Random Forest (the featured method in the original paper)

In [3]:
rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
rf.fit(X_train_s, y_train)
y_pred_rf = rf.predict(X_test_s)

acc_rf = accuracy_score(y_test, y_pred_rf)
mcc_rf = matthews_corrcoef(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)
print(f"Random Forest -> Accuracy: {acc_rf:.4f} | MCC: {mcc_rf:.4f} | F1: {f1_rf:.4f}")
print()
print(classification_report(y_test, y_pred_rf, target_names=['Survived', 'Death']))

Random Forest -> Accuracy: 0.8167 | MCC: 0.5563 | F1: 0.6667

              precision    recall  f1-score   support

    Survived       0.83      0.93      0.87        41
       Death       0.79      0.58      0.67        19

    accuracy                           0.82        60
   macro avg       0.81      0.75      0.77        60
weighted avg       0.81      0.82      0.81        60



## 3. Comparison with the Literature — `chicco2020machine`

| Model | Accuracy (original paper) | MCC (original paper) | Accuracy (this notebook) | MCC (this notebook) |
|---|:---:|:---:|:---:|:---:|
| Random Forest | 74.0% | 0.384 | *(see the output cell below — REAL numbers from this run, not hard-coded)* | *(idem)* |

> ⚠️ Methodological note: the original paper uses 10-fold cross-validation
> repeated 100 times and reports the average; this notebook (as required
> by Step 1) uses a single, fixed 80/20 Train/Test split with
> `random_state=42` — so a single run's result can fluctuate around the
> 74%/0.384 benchmark rather than matching it exactly. This limitation
> should be stated explicitly in the Step 5 report, not treated as a
> pipeline bug.

In [4]:
print("=== COMPARISON WITH THE LITERATURE (real numbers from this run) ===")
print(f"{'Model':<22}{'Accuracy':>12}{'MCC':>10}{'F1':>10}")
print(f"{'Logistic Regression':<22}{acc_lr:>12.4f}{mcc_lr:>10.4f}{f1_lr:>10.4f}")
print(f"{'Random Forest':<22}{acc_rf:>12.4f}{mcc_rf:>10.4f}{f1_rf:>10.4f}")
print(f"{'chicco2020machine (RF, paper)':<22}{'0.7400':>12}{'0.3840':>10}{'-':>10}")
delta_acc = acc_rf - 0.740
delta_mcc = mcc_rf - 0.384
print(f"\nDelta vs. the literature: Accuracy {delta_acc:+.4f} | MCC {delta_mcc:+.4f}")

=== COMPARISON WITH THE LITERATURE (real numbers from this run) ===
Model                     Accuracy       MCC        F1
Logistic Regression         0.8167    0.5563    0.6667
Random Forest               0.8167    0.5563    0.6667
chicco2020machine (RF, paper)      0.7400    0.3840         -

Delta vs. the literature: Accuracy +0.0767 | MCC +0.1723


## 4. Step 2 Summary

The Random Forest result on the held-out Test set (n=60, 20% of 299) is
reported honestly in the cell above — not rounded or adjusted to match
the literature. Any difference from the 74.0%/0.384 benchmark of
`chicco2020machine` stems from differing evaluation methodology (1-split
vs. 100×10-fold CV) and the small Test size (n=60), which inflates
estimate variance — exactly the limitation noted in Chapter 2 Section
2.10 ("small sample size, lack of external validation").

**Next:** [`3_improvement.ipynb`](./3_improvement.ipynb) — Step 3, applying
SMOTE toward the 92.6% benchmark of `ishaq2021improving`.